# CombinedDataCleaner
Combines the 8 CIC-IDS-2017 CSVs, cleans them, and exports CleanedData.csv
Run cell-by-cell in Jupyter (paste each # %% block into its own cell).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

RAWDATA_DIR = Path("/content/rawdata")
OUTPUT_PATH = Path("CleanedData.csv")


## 1. Load and combine all 8 CSVs
Column names in this dataset have inconsistent leading/trailing whitespace
across files, so we strip them before checking alignment.

In [ ]:
csv_files = sorted(RAWDATA_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} files:")
for f in csv_files:
    print(" -", f.name)

dfs = []
for f in csv_files:
    df = pd.read_csv(f, encoding="latin1", low_memory=False)
    df.columns = df.columns.str.strip()
    dfs.append(df)



Found 8 files:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


# Sanity check: all files should have identical column sets after stripping

In [ ]:
col_sets = [set(d.columns) for d in dfs]
assert all(cs == col_sets[0] for cs in col_sets), "Column mismatch across files!"

data = pd.concat(dfs, ignore_index=True)
print("Combined shape:", data.shape)

Combined shape: (1264108, 79)


## 2. Drop identifier / leakage columns
These describe *who* not *how* traffic behaves, so they don't generalize
to unseen IPs/flows and would leak identity instead of behavior patterns.

In [ ]:
leakage_cols = [
    "Flow ID", "Source IP", "Source Port",
    "Destination IP", "Timestamp"
]
data = data.drop(columns=[c for c in leakage_cols if c in data.columns])

## 3. Clean invalid values
`Flow Bytes/s` and `Flow Packets/s` contain inf/-inf where Flow Duration
was 0 (division by zero). Convert those to NaN first, then drop NaNs,
otherwise a plain dropna() misses the infinities.


In [ ]:
data = data.replace([np.inf, -np.inf], np.nan)
before = len(data)
data = data.dropna()
print(f"Dropped {before - len(data)} rows with NaN/inf")

Dropped 1569 rows with NaN/inf


In [ ]:
before = len(data)
data = data.drop_duplicates()
print(f"Dropped {before - len(data)} duplicate rows")

Dropped 117432 duplicate rows


## 4. Drop near-constant columns
Columns with only 1 unique value carry no signal for the model.


In [ ]:
label_col = "Label"
feature_cols = [c for c in data.columns if c != label_col]

nunique = data[feature_cols].nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
print("Dropping near-constant columns:", constant_cols)

data = data.drop(columns=constant_cols)
feature_cols = [c for c in feature_cols if c not in constant_cols]

Dropping near-constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


## 5. Clean labels
Strip whitespace and inspect class distribution before deciding
binary vs multiclass. Currently set up for MULTICLASS (attack type).
To switch to binary, uncomment the line below.

In [ ]:
data[label_col] = data[label_col].str.strip()
print(data[label_col].value_counts())

Label
BENIGN                          983420
DoS Hulk                        141086
FTP-Patator                       5931
DoS slowloris                     5385
DoS Slowhttptest                  5228
SSH-Patator                       1869
Web Attack ï¿½ Brute Force        1470
Web Attack ï¿½ XSS                 652
Infiltration                        36
Web Attack ï¿½ Sql Injection        21
PortScan                             7
Bot                                  2
Name: count, dtype: int64


## 6. Normalize numeric features
Scale all feature columns to [0, 1]. Label column is excluded.

In [ ]:
scaler = MinMaxScaler()
data[feature_cols] = scaler.fit_transform(data[feature_cols])

In [ ]:
data.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned data to {OUTPUT_PATH} — shape: {data.shape}")

Saved cleaned data to CleanedData.csv — shape: (1145107, 71)


In [ ]:
import joblib

joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")
print(f"Saved cleaned data to {OUTPUT_PATH} — shape: {data.shape}")
print("Saved scaler.pkl and feature_cols.pkl")

Saved cleaned data to CleanedData.csv — shape: (1145098, 71)
Saved scaler.pkl and feature_cols.pkl
